In [3]:
import os
import re
import glob
import math
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt


# ----------------------------
# Config dos grupos desejados
# ----------------------------
KEYWORDS = [
    "TFM-PREV3",
    "TIMESFM",
    "BASELINE",
    "TFM-GAMMA85",
    "TFM-GAMMA95",
    "TFM-GAMMA75",
]

VARIANT_ORDER = [
    "TFM-PREV3",
    "TIMESFM",
    "BASELINE",
    "TFM-GAMMA85",
    "TFM-GAMMA95",
    "TFM-GAMMA75",
]

def format_x_1e5(x, pos):
    if x == 0:
        return "0"
    return f"{int(x/1e5)}×10⁵"

# ----------------------------
# Tree printer
# ----------------------------
def print_tree_dir(path, prefix=""):
    items = sorted(os.listdir(path))
    for i, item in enumerate(items):
        full = os.path.join(path, item)
        connector = "├── " if i < len(items) - 1 else "└── "
        print(prefix + connector + item)
        if os.path.isdir(full):
            new_prefix = prefix + ("│   " if i < len(items) - 1 else "    ")
            print_tree_dir(full, new_prefix)


# ----------------------------
# Discovery helpers
# ----------------------------
def find_experiments(root):
    """Experiment dirs containing progress.csv (walk recursively)."""
    exps = []
    for r, _, files in os.walk(root):
        if "progress.csv" in files:
            exps.append(r)
    return sorted(exps)


def extract_group_and_variant(name):
    """
    Examples
    --------
    'COMA-DefaultReward (0.5,0.7,0.5,0.6)'
        -> algo='COMA', variant='DefaultReward', group='COMA-DefaultReward'

    'MATRPO-RewardMaxMedian (0.5,0.7,0.5,0.6) - 3'
        -> algo='MATRPO', variant='RewardMaxMedian', group='MATRPO-RewardMaxMedian'

    Retorna (None, None, None) se não for um dos grupos desejados.
    """
    base = name.split(" - ")[0] if " - " in name else name

    if "-" not in base:
        return None, None, None

    algo = base.split("-", 1)[0].strip()

    variant = None
    for k in sorted(KEYWORDS, key=len, reverse=True):
        if k in base:
            variant = k
            break

    if variant is None:
        return None, None, None

    group = f"{algo}-{variant}"
    return algo, variant, group


def is_target_experiment(name):
    algo, variant, group = extract_group_and_variant(name)
    return group is not None

# ---------------------------------------------------
# Episode metrics (média das runs)
# ---------------------------------------------------
def compute_episode_metrics(metrics_dir, results_dir):
    import glob
    import re

    files = glob.glob(os.path.join(metrics_dir, "episode_metrics_*_run*.csv"))

    if not files:
        print("[INFO] Nenhum episode_metrics encontrado.")
        return None

    groups = defaultdict(list)

    pattern = re.compile(r"episode_metrics_(.*?)_run\d+\.csv")

    for f in files:
        name = os.path.basename(f)

        m = pattern.match(name)
        if not m:
            continue

        algo = m.group(1)

        try:
            df = pd.read_csv(f)
        except Exception:
            continue

        groups[algo].append(df)

    rows = []

    for algo, dfs in sorted(groups.items()):

        cols = [c for c in dfs[0].columns if c != "episode"]

        values = {}

        for c in cols:

            medias = []

            for df in dfs:

                if c not in df.columns:
                    continue

                medias.append(df[c].mean())

            values[c] = np.mean(medias)

        values["algorithm"] = algo
        values["runs"] = len(dfs)

        rows.append(values)

    result = pd.DataFrame(rows)

    cols = ["algorithm", "runs"] + [c for c in result.columns if c not in ["algorithm", "runs"]]
    result = result[cols]

    out = os.path.join(results_dir, "episode_metrics_average.csv")
    result.to_csv(out, index=False)

    print("\n✅ Episode metrics salvos em:")
    print(out)

    try:
        from IPython.display import display
        display(result)
    except:
        print(result)

    return result


# ----------------------------
# Episode metrics
# ----------------------------
def compute_episode_metrics(root):

    metrics_dir = os.path.join(root, "metrics")

    if not os.path.isdir(metrics_dir):
        empty_grp = pd.DataFrame(
            columns=["algo", "variant", "metric", "mean", "std", "count", "mean_std"]
        )
        empty_df = pd.DataFrame()
        return empty_grp, empty_df

    rows = []

    pattern = re.compile(r"episode_metrics_(.*?)_run(\d+)\.csv")

    for file in glob.glob(os.path.join(metrics_dir, "episode_metrics_*_run*.csv")):

        name = os.path.basename(file)

        m = pattern.match(name)
        if not m:
            continue

        algo = m.group(1)
        run = f"run{m.group(2)}"

        # usa o nome da pasta (baseline, timesfm, tfm_prev3)
        scenario = os.path.basename(root).upper()

        variant_map = {
            "BASELINE": "BASELINE",
            "TIMESFM": "TIMESFM",
            "TFM_PREV3": "TFM_PREV3",
            "TFM-GAMMA85": "TFM-GAMMA85",
            "TFM-GAMMA95": "TFM-GAMMA95",
            "TFM-GAMMA75": "TFM-GAMMA75",
        }

        variant = variant_map.get(scenario, scenario)

        try:
            df = pd.read_csv(file)
        except Exception:
            continue

        numeric_cols = [
            c for c in df.columns
            if c != "episode"
        ]

        row = {
            "algo": algo,
            "variant": variant,
            "group": f"{algo}-{variant}",
            "seed": run
        }

        for c in numeric_cols:
            row[c] = pd.to_numeric(df[c], errors="coerce").mean()

        rows.append(row)

    metrics_df = pd.DataFrame(rows)

    if metrics_df.empty:

        metrics_grp = pd.DataFrame(
            columns=[
                "algo",
                "variant",
                "metric",
                "mean",
                "std",
                "count",
                "mean_std",
            ]
        )

        return metrics_grp, metrics_df

    grouped_rows = []

    metric_cols = [
        c
        for c in metrics_df.columns
        if c not in ["algo", "variant", "group", "seed"]
    ]

    for metric in metric_cols:

        grp = (
            metrics_df
            .groupby(["algo", "variant"])[metric]
            .agg(["mean", "std", "count"])
            .reset_index()
        )

        grp["metric"] = metric

        grp["mean_std"] = (
            grp["mean"].round(3).astype(str)
            + " ± "
            + grp["std"].round(3).astype(str)
        )

        grouped_rows.append(grp)

    metrics_grp = pd.concat(grouped_rows, ignore_index=True)

    metrics_grp["variant"] = pd.Categorical(
        metrics_grp["variant"],
        categories=VARIANT_ORDER,
        ordered=True,
    )

    metrics_grp = metrics_grp.sort_values(
        ["algo", "variant", "metric"]
    ).reset_index(drop=True)

    return metrics_grp, metrics_df



# ----------------------------
# Reward + AUC from progress.csv
# ----------------------------
def compute_reward_and_auc(root, reward_col="episode_reward_mean"):
    reward_rows, auc_rows = [], []

    for exp in find_experiments(root):
        seed = os.path.basename(exp)

        algo, variant, group = extract_group_and_variant(seed)
        if group is None:
            continue

        csv = os.path.join(exp, "progress.csv")
        try:
            df = pd.read_csv(csv)
        except Exception:
            continue

        if reward_col not in df.columns:
            continue

        rewards = pd.to_numeric(df[reward_col], errors="coerce").dropna()
        if rewards.empty:
            continue

        # ----------------------------
        # FINAL REWARD (igual antes)
        # ----------------------------
        reward_rows.append({
            "algo": algo,
            "variant": variant,
            "group": group,
            "seed": seed,
            "mean_reward": float(rewards.mean())
        })

        # ============================================================
        # 🔥 AUC NORMALIZADO (ESTILO PAPER - TABELA 3)
        # ============================================================

        r = rewards.values.astype(float)

        r_min = r.min()
        r_max = r.max()

        # evita divisão por zero
        if (r_max - r_min) < 1e-8:
            continue

        # 1. normalização 0–1  <<< ALTERADO
        r_norm = (r - r_min) / (r_max - r_min)

        # 2. eixo do tempo (índice) <<< ALTERADO
        x = np.arange(len(r_norm))

        # 3. AUC normalizado <<< ALTERADO
        auc = np.trapz(r_norm, x) / (x[-1] - x[0])

        auc_rows.append({
            "algo": algo,
            "variant": variant,
            "group": group,
            "seed": seed,
            "auc": float(auc)
        })

    # ----------------------------
    # (RESTO DA FUNÇÃO NÃO MUDA)
    # ----------------------------

    reward_df = pd.DataFrame(reward_rows)
    auc_df = pd.DataFrame(auc_rows)

    if reward_df.empty:
        reward_grp = pd.DataFrame(columns=["algo", "variant", "mean", "std", "count", "mean_std"])
    else:
        reward_grp = (
            reward_df
            .groupby(["algo", "variant"])["mean_reward"]
            .agg(["mean", "std", "count"])
            .reset_index()
        )
        reward_grp["variant"] = pd.Categorical(
            reward_grp["variant"],
            categories=VARIANT_ORDER,
            ordered=True
        )
        reward_grp = reward_grp.sort_values(["algo", "variant"]).reset_index(drop=True)
        reward_grp["mean_std"] = (
            reward_grp["mean"].round(3).astype(str)
            + " ± "
            + reward_grp["std"].round(3).astype(str)
        )

    if auc_df.empty:
        auc_grp = pd.DataFrame(columns=["algo", "variant", "mean", "std", "count", "mean_std"])
    else:
        auc_grp = (
            auc_df
            .groupby(["algo", "variant"])["auc"]
            .agg(["mean", "std", "count"])
            .reset_index()
        )
        auc_grp["variant"] = pd.Categorical(
            auc_grp["variant"],
            categories=VARIANT_ORDER,
            ordered=True
        )
        auc_grp = auc_grp.sort_values(["algo", "variant"]).reset_index(drop=True)
        auc_grp["mean_std"] = (
            auc_grp["mean"].round(3).astype(str)
            + " ± "
            + auc_grp["std"].round(3).astype(str)
        )

    return reward_grp, auc_grp, reward_df, auc_df


# ----------------------------
# TensorBoard tag utilities
# ----------------------------
RAY_PREFIX = re.compile(r"^(ray/(tune|train|rllib)/)")


def strip_ray(tag):
    return RAY_PREFIX.sub("", tag)


REWARD_PREF = [
    "episode_reward_mean",
]

REWARD_REGEX = re.compile(r"(episode|ep).*(reward|return).*(mean|avg)", re.IGNORECASE)

X_PREF = [
    "timesteps_total",
    "num_env_steps_sampled",
    "num_agent_steps_sampled",
    "training_iteration",
]


def choose_reward_tag(tags):
    for t in REWARD_PREF:
        if t in tags:
            return t
    for t in tags:
        if REWARD_REGEX.search(t):
            return t
    return None


def choose_x_tag(tags):
    for t in X_PREF:
        if t in tags:
            return t
    return None


def get_event_files(d):
    return sorted(glob.glob(os.path.join(d, "**", "events.out.tfevents.*"), recursive=True))


# ----------------------------
# Smoothing + plotting
# ----------------------------
def _smooth(arr, sigma):
    if sigma is None or sigma <= 0:
        return arr
    try:
        from scipy.ndimage import gaussian_filter1d
        return gaussian_filter1d(arr, sigma=sigma)
    except Exception:
        w = int(max(3, round(2 * sigma + 1)))
        if w % 2 == 0:
            w += 1
        kernel = np.ones(w) / w
        return np.convolve(arr, kernel, mode="same")


def _safe_minmax(values):
    """
    GUARANTEE:
      - vmin = min(values)
      - vmax = max(values)
      - if degenerate, widen slightly
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 0.0, 1.0

    vmin = float(np.min(values))
    vmax = float(np.max(values))

    if not np.isfinite(vmin) or not np.isfinite(vmax) or (vmax - vmin) < 1e-12:
        vmin -= 1.0
        vmax += 1.0

    return vmin, vmax


def _norm01(y, vmin, vmax):
    denom = (vmax - vmin) if (vmax - vmin) != 0 else 1.0
    return np.clip((y - vmin) / denom, 0.0, 1.0)


def plot_raw(xs, m, s, title, save, smooth_sigma):
    import matplotlib.pyplot as plt

    m2, s2 = _smooth(m, smooth_sigma), _smooth(s, smooth_sigma)

    plt.figure(figsize=(20, 10))
    ax = plt.gca()

    plt.plot(xs, m2, label="episode_reward_mean")
    plt.fill_between(xs, m2 - s2, m2 + s2, alpha=0.2)

    ax.set_xlabel("Step", fontsize=30)
    ax.set_ylabel("Reward", fontsize=30)
    ax.tick_params(axis='both', labelsize=26)

    plt.legend(fontsize=24)
    plt.grid(True)

    os.makedirs(os.path.dirname(save), exist_ok=True)
    plt.savefig(save, dpi=300, bbox_inches="tight")
    plt.close()


def plot_norm(xs, m, s, title, save, smooth_sigma, vmin, vmax):
    import matplotlib.pyplot as plt

    m2, s2 = _smooth(m, smooth_sigma), _smooth(s, smooth_sigma)

    nm = _norm01(m2, vmin, vmax)
    denom = (vmax - vmin) if (vmax - vmin) != 0 else 1.0
    ns = s2 / denom
    lower = np.clip(nm - ns, 0, 1)
    upper = np.clip(nm + ns, 0, 1)

    plt.figure(figsize=(20, 10))
    ax = plt.gca()

    max_x = xs.max()
    ticks = np.arange(0, max_x + 1e5, 1e5)
    ax.set_xticks(ticks)
    ax.xaxis.set_major_formatter(FuncFormatter(format_x_1e5))

    plt.plot(xs, nm, label="episode_reward_mean")
    plt.fill_between(xs, lower, upper, alpha=0.2)
    plt.ylim(0, 1)

    ax.set_xlabel("Step", fontsize=30)
    ax.set_ylabel("Reward", fontsize=30)
    ax.tick_params(axis='both', labelsize=26)

    plt.legend(fontsize=26)
    plt.grid(True)

    os.makedirs(os.path.dirname(save), exist_ok=True)
    plt.savefig(save, dpi=150, bbox_inches="tight")
    plt.close()

# ----------------------------
# Charts: agrupados por algoritmo + keyword
# ----------------------------
def make_charts(
    root,
    results_dir,
    smooth_sigma=3.0,
    norm_scope="global",  # "global" or "per_group"
    max_x=None,
):
    """
    Guarantees:
      - Normalized plot uses MIN-MAX
      - 0 == lowest value, 1 == highest value
      - scope controls whether min/max is computed globally or per group

    Grouping:
      - one chart group per "{algo}-{variant}"
      - examples:
          COMA-DefaultReward
          COMA-RewardHard
          MATRPO-RewardMaxMedian

    X axis:
      - uses x_tag if found (timesteps_total, training_iteration, ...)
      - otherwise uses event.step
    """
    try:
        from tensorflow.python.summary.summary_iterator import summary_iterator
    except Exception as e:
        print(f"[WARN] TensorFlow not available — skipping charts ({e})")
        return

    groups = defaultdict(list)

    for exp in find_experiments(root):
        seed = os.path.basename(exp)

        algo, variant, group = extract_group_and_variant(seed)
        if group is None:
            continue

        groups[group].append(exp)

    if not groups:
        print("[WARN] No matching experiment folders found (with progress.csv).")
        return

    def extract_group_series(seed_dirs):
        event_files = []
        for sd in seed_dirs:
            event_files.extend(get_event_files(sd))
        if not event_files:
            return None

        tag_counts = Counter()
        for ef in event_files:
            try:
                for e in summary_iterator(ef):
                    if not hasattr(e, "summary") or e.summary is None:
                        continue
                    for v in e.summary.value:
                        tag_counts[strip_ray(v.tag)] += 1
            except Exception:
                pass

        y_tag = choose_reward_tag(tag_counts.keys())
        if not y_tag:
            return None

        x_tag = choose_x_tag(tag_counts.keys())

        y_by_step = defaultdict(list)
        x_by_step = defaultdict(list) if x_tag else None

        for ef in event_files:
            try:
                for e in summary_iterator(ef):
                    if not hasattr(e, "summary") or e.summary is None:
                        continue
                    step = int(e.step)
                    for v in e.summary.value:
                        tag = strip_ray(v.tag)
                        if x_tag and tag == x_tag:
                            x_by_step[step].append(float(v.simple_value))
                        if tag == y_tag:
                            y_by_step[step].append(float(v.simple_value))
            except Exception:
                pass

        if not y_by_step:
            return None

        steps = sorted(y_by_step.keys())

        if x_tag:
            xs = np.array(
                [
                    np.mean(x_by_step[s]) if s in x_by_step and len(x_by_step[s]) > 0 else float(s)
                    for s in steps
                ],
                dtype=float
            )
        else:
            xs = np.array([float(s) for s in steps], dtype=float)

        m = np.array([np.mean(y_by_step[s]) for s in steps], dtype=float)
        s = np.array([np.std(y_by_step[s]) for s in steps], dtype=float)

        order = np.argsort(xs)
        xs = xs[order]
        m = m[order]
        s = s[order]

        if max_x is not None:
            mask = xs <= max_x
            xs = xs[mask]
            m = m[mask]
            s = s[mask]

        if len(xs) == 0:
            return None

        return xs, m, s, y_tag, x_tag

    cached = {}
    global_smoothed_means = []

    for group, seed_dirs in sorted(groups.items()):
        triple = extract_group_series(seed_dirs)
        if triple is None:
            continue

        xs, m, s, y_tag, x_tag = triple
        m_sm = _smooth(m, smooth_sigma)

        cached[group] = (xs, m, s, m_sm, y_tag, x_tag)
        global_smoothed_means.extend(m_sm.tolist())

    if not cached:
        print("[WARN] No chartable data found (no tfevents or no reward tag).")
        return

    global_vmin, global_vmax = _safe_minmax(np.array(global_smoothed_means))

    print("[Charts] X auto (x_tag if exists else step).")
    print(f"[Charts] Normalization = MIN-MAX with scope={norm_scope}.")
    print(f"[Charts] Global min-max (on smoothed means): vmin={global_vmin:.6f}, vmax={global_vmax:.6f}")

    charts_root = os.path.join(results_dir, "charts")

    for group, (xs, m, s, m_sm, y_tag, x_tag) in cached.items():
        out_dir = os.path.join(charts_root, group)

        if norm_scope == "per_group":
            vmin, vmax = _safe_minmax(m_sm)
        else:
            vmin, vmax = global_vmin, global_vmax

        plot_raw(
            xs, m, s,
            title=f"{group} — raw (x={x_tag or 'step'}, y={y_tag})",
            save=os.path.join(out_dir, "raw.png"),
            smooth_sigma=smooth_sigma
        )

        plot_norm(
            xs, m, s,
            title=f"{group} — normalized (x={x_tag or 'step'}, y={y_tag})",
            save=os.path.join(out_dir, "normalized.png"),
            smooth_sigma=smooth_sigma,
            vmin=vmin,
            vmax=vmax
        )

        print(
            f"✔ charts: {group} | x={x_tag or 'step'} | y={y_tag} "
            f"| norm vmin={vmin:.6f} vmax={vmax:.6f}"
        )

    n = len(cached)

    cols = 3
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(7*cols, 4.5*rows)
    )

    axes = axes.flatten()

    for ax, (group, data) in zip(axes, cached.items()):

        xs, m, s, m_sm, y_tag, x_tag = data

        if norm_scope == "per_group":
            vmin, vmax = _safe_minmax(m_sm)
        else:
            vmin, vmax = global_vmin, global_vmax

        nm = _norm01(m_sm, vmin, vmax)

        ax.plot(xs, nm)

        ax.set_ylim(0,1)

        ax.grid(True)

        algo, variant = group.split("-", 1)

        variant_map = {
            "BASELINE": "BASELINE",
            "TIMESFM": "TIMESFM",
            "TFM_PREV3": "TFM_PREV3",
            "TFM-GAMMA85": "TFM-GAMMA85",
            "TFM-GAMMA95": "TFM-GAMMA95",
            "TFM-GAMMA75": "TFM-GAMMA75",
        }

        ax.set_title(
            f"{algo.upper()} - {variant_map.get(variant, variant)}",
            fontsize=11,
            fontweight="bold"
        )

    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()

    plt.savefig(
        os.path.join(charts_root, "comparison_normalized.png"),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    fig, axes = plt.subplots(rows, cols, figsize=(7*cols, 4.5*rows))
    axes = axes.flatten()

    for ax, (group, data) in zip(axes, cached.items()):

        xs, m, s, m_sm, y_tag, x_tag = data

        ax.plot(xs, m_sm, linewidth=2)

        ax.grid(True)

        algo, variant = group.split("-", 1)

        variant_map = {
            "BASELINE": "BASELINE",
            "TIMESFM": "TIMESFM",
            "TFM_PREV3": "TFM_PREV3",
            "TFM-GAMMA85": "TFM-GAMMA85",
            "TFM-GAMMA95": "TFM-GAMMA95",
            "TFM-GAMMA75": "TFM-GAMMA75",
        }

        ax.set_title(
            f"{algo.upper()} - {variant_map.get(variant, variant)}",
            fontsize=11,
            fontweight="bold"
        )

        ax.set_xlabel(x_tag)
        ax.set_ylabel(y_tag)

    for ax in axes[len(cached):]:
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(
        os.path.join(charts_root, "comparison_raw.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

# ----------------------------
# MAIN ENTRY (Notebook-safe)
# ----------------------------
def run_all(
    root_folder,
    results_folder_name=None,
    reward_col="episode_reward_mean",
    print_tree=True,
    do_charts=True,
    smooth_sigma=3.0,
    norm_scope="global",  # "global" or "per_group"
    show_seed_tables=False,
    max_x=None,
):
    root_folder = os.path.abspath(root_folder)
    if not os.path.isdir(root_folder):
        raise ValueError(f"Folder not found: {root_folder}")

    base = os.path.basename(os.path.normpath(root_folder))
    if results_folder_name is None:
        results_folder_name = f"{base}_results"

    results_dir = os.path.join(os.path.dirname(root_folder), results_folder_name)
    os.makedirs(results_dir, exist_ok=True)

    if print_tree:
        print("📂 Folder tree:")
        print_tree_dir(root_folder)

    reward_grp_all = []
    auc_grp_all = []

    reward_seed_all = []
    auc_seed_all = []

    metrics_grp_all = []
    metrics_seed_all = []

    for scenario in ["baseline", "timesfm", "tfm_Prev3", "tfm_gamma85", "tfm_gamma95", "tfm_gamma75"]:

        folder = os.path.join(root_folder, scenario)

        if not os.path.isdir(folder):
            continue

        print(f"\n==============================")
        print(f"Processando {scenario}")
        print("==============================")

        rg, ag, rs, aus = compute_reward_and_auc(folder, reward_col)

        rg["scenario"] = scenario
        ag["scenario"] = scenario
        rs["scenario"] = scenario
        aus["scenario"] = scenario

        reward_grp_all.append(rg)
        auc_grp_all.append(ag)
        reward_seed_all.append(rs)
        auc_seed_all.append(aus)

        mg, ms = compute_episode_metrics(folder)

        mg["scenario"] = scenario
        ms["scenario"] = scenario

        metrics_grp_all.append(mg)
        metrics_seed_all.append(ms)

    reward_grp = pd.concat(reward_grp_all, ignore_index=True)
    auc_grp = pd.concat(auc_grp_all, ignore_index=True)
    reward_seeds = pd.concat(reward_seed_all, ignore_index=True)
    auc_seeds = pd.concat(auc_seed_all, ignore_index=True)
    
    metrics_grp = pd.concat(metrics_grp_all, ignore_index=True)
    metrics_seed = pd.concat(metrics_seed_all, ignore_index=True)

    reward_csv = os.path.join(results_dir, "reward_stats_grouped.csv")
    auc_csv = os.path.join(results_dir, "auc_grouped.csv")
    reward_seed_csv = os.path.join(results_dir, "reward_stats_per_seed.csv")
    auc_seed_csv = os.path.join(results_dir, "auc_per_seed.csv")

    metrics_grp_csv = os.path.join(results_dir, "episode_metrics_grouped.csv")
    metrics_seed_csv = os.path.join(results_dir, "episode_metrics_per_seed.csv")

    metrics_grp.to_csv(metrics_grp_csv, index=False)
    metrics_seed.to_csv(metrics_seed_csv, index=False)

    reward_grp.to_csv(reward_csv, index=False)
    auc_grp.to_csv(auc_csv, index=False)
    reward_seeds.to_csv(reward_seed_csv, index=False)
    auc_seeds.to_csv(auc_seed_csv, index=False)

    print("\n✅ Saved:")
    print(reward_csv)
    print(auc_csv)
    print(reward_seed_csv)
    print(auc_seed_csv)

    try:
        from IPython.display import display

        display(reward_grp)
        display(auc_grp)

        if show_seed_tables:
            if not reward_seeds.empty:
                reward_seeds = reward_seeds.sort_values(["algo", "variant", "seed"])
            if not auc_seeds.empty:
                auc_seeds = auc_seeds.sort_values(["algo", "variant", "seed"])

            display(reward_seeds)
            display(auc_seeds)

    except Exception:
        print("\n[Reward grouped]\n", reward_grp)
        print("\n[AUC grouped]\n", auc_grp)

        if show_seed_tables:
            print("\n[Reward per seed]\n", reward_seeds.sort_values(["algo", "variant", "seed"]))
            print("\n[AUC per seed]\n", auc_seeds.sort_values(["algo", "variant", "seed"]))

    if do_charts:

        for scenario in ["baseline", "timesfm", "tfm_Prev3", "tfm_gamma85", "tfm_gamma95", "tfm_gamma75"]:

            folder = os.path.join(root_folder, scenario)

            if not os.path.isdir(folder):
                continue

            print(f"\n📊 Gerando gráficos para {scenario}")

            make_charts(
                root=folder,
                results_dir=os.path.join(results_dir, scenario),
                smooth_sigma=smooth_sigma,
                norm_scope=norm_scope,
                max_x=max_x
            )

        print("\n📊 Charts gerados.")

    metrics_dir = os.path.join(root_folder, "metrics")

    if os.path.isdir(metrics_dir):
        compute_episode_metrics(metrics_dir, results_dir)
    print("\n[DONE] Results folder:", results_dir)
    return results_dir

In [4]:
run_all(
    root_folder="/mnt/ssd1/wesley/BusEnv/1runs",
    results_folder_name="runs_timesfm",
    reward_col="episode_reward_mean",
    print_tree=False,
    do_charts=True,
    smooth_sigma=3.0,
    norm_scope="global",
    show_seed_tables=True,
    max_x=None
)



Processando baseline



Processando timesfm

Processando tfm_Prev3

Processando tfm_gamma85

Processando tfm_gamma95

Processando tfm_gamma75

✅ Saved:
/mnt/ssd1/wesley/BusEnv/runs_timesfm/reward_stats_grouped.csv
/mnt/ssd1/wesley/BusEnv/runs_timesfm/auc_grouped.csv
/mnt/ssd1/wesley/BusEnv/runs_timesfm/reward_stats_per_seed.csv
/mnt/ssd1/wesley/BusEnv/runs_timesfm/auc_per_seed.csv


,algo,variant,mean,std,count,mean_std,scenario
0,happo,BASELINE,1389.213095,0.436049,5,1389.213 ± 0.436,baseline
1,hatrpo,BASELINE,2457.325097,29.808688,5,2457.325 ± 29.809,baseline
2,ia2c,BASELINE,3495.526079,4.637850,5,3495.526 ± 4.638,baseline
3,ippo,BASELINE,4032.626705,0.000000,5,4032.627 ± 0.0,baseline
4,itrpo,BASELINE,3333.834293,0.000000,5,3333.834 ± 0.0,baseline
5,maa2c,BASELINE,3472.063368,8.305613,5,3472.063 ± 8.306,baseline
6,mappo,BASELINE,3990.994669,0.000000,5,3990.995 ± 0.0,baseline
7,matrpo,BASELINE,3457.274383,0.000000,5,3457.274 ± 0.0,baseline
8,happo,TIMESFM,-175.763381,0.162774,5,-175.763 ± 0.163,timesfm
9,hatrpo,TIMESFM,-35.304845,4.858510,5,-35.305 ± 4.859,timesfm


,algo,variant,mean,std,count,mean_std,scenario
0,happo,BASELINE,0.883844,0.005924,5,0.884 ± 0.006,baseline
1,hatrpo,BASELINE,0.741055,0.032236,5,0.741 ± 0.032,baseline
2,ia2c,BASELINE,0.761728,0.001991,5,0.762 ± 0.002,baseline
3,ippo,BASELINE,0.887992,0.000000,5,0.888 ± 0.0,baseline
4,itrpo,BASELINE,0.732899,0.000000,5,0.733 ± 0.0,baseline
5,maa2c,BASELINE,0.745563,0.002933,5,0.746 ± 0.003,baseline
6,mappo,BASELINE,0.863287,0.000000,5,0.863 ± 0.0,baseline
7,matrpo,BASELINE,0.715950,0.000000,5,0.716 ± 0.0,baseline
8,happo,TIMESFM,0.174669,0.008519,5,0.175 ± 0.009,timesfm
9,hatrpo,TIMESFM,0.807586,0.029727,5,0.808 ± 0.03,timesfm


,algo,variant,group,seed,mean_reward,scenario
0,happo,BASELINE,happo-BASELINE,happo-BASELINE-1,1389.000161,baseline
1,happo,BASELINE,happo-BASELINE,happo-BASELINE-2,1389.036286,baseline
2,happo,BASELINE,happo-BASELINE,happo-BASELINE-3,1389.823313,baseline
3,happo,BASELINE,happo-BASELINE,happo-BASELINE-4,1388.723858,baseline
4,happo,BASELINE,happo-BASELINE,happo-BASELINE-5,1389.481857,baseline
...,...,...,...,...,...,...
115,matrpo_TFM,TFM-PREV3,matrpo_TFM-TFM-PREV3,matrpo_TFM-PREV3-1,-106.704175,tfm_Prev3
116,matrpo_TFM,TFM-PREV3,matrpo_TFM-TFM-PREV3,matrpo_TFM-PREV3-2,-106.704175,tfm_Prev3
117,matrpo_TFM,TFM-PREV3,matrpo_TFM-TFM-PREV3,matrpo_TFM-PREV3-3,-106.704175,tfm_Prev3
118,matrpo_TFM,TFM-PREV3,matrpo_TFM-TFM-PREV3,matrpo_TFM-PREV3-4,-106.704175,tfm_Prev3


,algo,variant,group,seed,auc,scenario
0,happo,BASELINE,happo-BASELINE,happo-BASELINE-1,0.890349,baseline
1,happo,BASELINE,happo-BASELINE,happo-BASELINE-2,0.874735,baseline
2,happo,BASELINE,happo-BASELINE,happo-BASELINE-3,0.882979,baseline
3,happo,BASELINE,happo-BASELINE,happo-BASELINE-4,0.883541,baseline
4,happo,BASELINE,happo-BASELINE,happo-BASELINE-5,0.887617,baseline
...,...,...,...,...,...,...
115,matrpo_TFM,TFM-PREV3,matrpo_TFM-TFM-PREV3,matrpo_TFM-PREV3-1,0.844585,tfm_Prev3
116,matrpo_TFM,TFM-PREV3,matrpo_TFM-TFM-PREV3,matrpo_TFM-PREV3-2,0.844585,tfm_Prev3
117,matrpo_TFM,TFM-PREV3,matrpo_TFM-TFM-PREV3,matrpo_TFM-PREV3-3,0.844585,tfm_Prev3
118,matrpo_TFM,TFM-PREV3,matrpo_TFM-TFM-PREV3,matrpo_TFM-PREV3-4,0.844585,tfm_Prev3



📊 Gerando gráficos para baseline
[Charts] X auto (x_tag if exists else step).
[Charts] Normalization = MIN-MAX with scope=global.
[Charts] Global min-max (on smoothed means): vmin=1309.157015, vmax=4428.612459
✔ charts: happo-BASELINE | x=timesteps_total | y=episode_reward_mean | norm vmin=1309.157015 vmax=4428.612459
✔ charts: hatrpo-BASELINE | x=timesteps_total | y=episode_reward_mean | norm vmin=1309.157015 vmax=4428.612459
✔ charts: ia2c-BASELINE | x=timesteps_total | y=episode_reward_mean | norm vmin=1309.157015 vmax=4428.612459
✔ charts: ippo-BASELINE | x=timesteps_total | y=episode_reward_mean | norm vmin=1309.157015 vmax=4428.612459
✔ charts: itrpo-BASELINE | x=timesteps_total | y=episode_reward_mean | norm vmin=1309.157015 vmax=4428.612459
✔ charts: maa2c-BASELINE | x=timesteps_total | y=episode_reward_mean | norm vmin=1309.157015 vmax=4428.612459
✔ charts: mappo-BASELINE | x=timesteps_total | y=episode_reward_mean | norm vmin=1309.157015 vmax=4428.612459
✔ charts: matrpo-BAS

'/mnt/ssd1/wesley/BusEnv/runs_timesfm'